In [ ]:
target_layer_idx = 16
activations = {}

def hook_fn(module, input, output):
    activations['features'] = output

handle = vgg.features[target_layer_idx].register_forward_hook(hook_fn)

batch_size = 32
all_features = []

with torch.no_grad():
    for i in range(0, len(img_stck), batch_size):
        batch_imgs = img_stck[i:i+batch_size]
        batch_tensors = torch.stack([preprocess(img.astype(np.uint8)) for img in batch_imgs])
        vgg(batch_tensors)
        feats = activations['features']
        feats_pooled = feats.mean(dim=[2, 3])
        all_features.append(feats_pooled.numpy())

handle.remove()
vgg_features = np.concatenate(all_features, axis=0)
print("VGG features shape:", vgg_features.shape)  # expect (540, 256)

In [ ]:
print("Any NaNs?", np.isnan(vgg_features).any())
print("Min/Max:", vgg_features.min(), vgg_features.max())
print("Mean activation:", vgg_features.mean())

In [ ]:
X = vgg_features
y = mean_centered_response.T
print("X shape", X.shape)
print("y shape", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2,random_state=42)
print("X test shape", X_test.shape)
print("y test shape", y_test.shape)
print("X train shape", X_train.shape)
print("y train shape", y_train.shape)

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
model = Ridge(alpha = 1000.0)
model.fit(X_train,y_train)
y_pred = model.predict(X_test)
print("y_pred shape", y_pred.shape)

In [ ]:
from sklearn.linear_model import RidgeCV

# Try a range of alpha values, let cross-validation pick the best one
alphas = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 500000.0]
model = RidgeCV(alphas=alphas, store_cv_results=True)
model.fit(X_train, y_train)

print("Best alpha found:", model.alpha_)

In [ ]:
from sklearn.metrics import explained_variance_score, r2_score
per_neuron_ev = explained_variance_score(y_test, y_pred, multioutput='raw_values')

print("explained_variance_per_neuron")
for i,ev in enumerate(per_neuron_ev):
  print(f"Neuron {i}: {ev:.3f}")
print("\nMean explained variance across all neurons", per_neuron_ev.mean())
print("best neuron:", per_neuron_ev.argmax(),"with ev:", per_neuron_ev.max())
print("worst neuron:", per_neuron_ev.argmin(),"with ev:",per_neuron_ev.min())

In [ ]:
# Get the actual integer positions of the centered neurons (not just the boolean mask)
centered_indices = np.where(cen_neurons)[0]
print("Indices of centered neurons:", centered_indices)
print("Number of indices:", len(centered_indices))

# Check if these indices themselves contain duplicates (they shouldn't, from a boolean mask)
print("Are indices unique?", len(centered_indices) == len(set(centered_indices)))

# Now look at the RF_SPATIAL rows at exactly these indices, raw, no intermediate variables
for idx in centered_indices:
    print(idx, mat['RF_SPATIAL'][idx, :2])  # just x,y for brevity

duplicate neurons found in the datset out of 26 only 15 are unique ones rest 11 are either duplicates or extremly similar to the other ones

In [ ]:

import numpy as np
np.save('vgg_features.npy', vgg_features)
np.save('mean_response_centered.npy', mean_response_centered)
np.save('centered_neurons.npy', centered_neurons)